**Cell 1**

# NB15 — Assistant task: geometry of the supervisor's PPO experts

**Purpose.** Before any text is generated, compute the relationship matrix `R` of the three
assistant-task experts (helpful, harmless, humor; Llama-2-7B, PPO, LoRA, seed 8888) and the
floor certificate. With m = 3 the floor can open in principle: it opens exactly when some entry
of `R⁻¹·1` is non-positive, i.e. one expert is closer to both others than they are to each
other (redundancy). This was never the case on HelpSteer2.

**Part A (always, no GPU, no token):** find and audit the three adapters, compute `R_cos`,
`R_gram`, norms, and the floor LP for 74 preferences (10 named + 64 Dirichlet, seed 137) on both
matrices. Script: `scripts/assistant_adapter_geometry.py`, which uses the same geometry code as
NB13/NB14 (`src/effective_lora_geometry.py`).

**Part B (optional smoke test, A100 + `HF_TOKEN`):** load the raw `meta-llama/Llama-2-7b-hf`
exactly as in the supervisor's snippet, attach each expert, and generate three short answers
(128 new tokens, greedy, `\n\nHuman: … \n\nAssistant:`). This confirms that each folder holds
the expert its name claims (the harmless expert should refuse the lock-picking prompt).

**Inputs.** The supervisor's Drive folder with its structure intact, i.e. three folders
`helpful_model`, `harmless_model`, `humor_model` (as in the shared Drive folder) or
`ppo_assistant_<goal>_s8888/…/best_model`, somewhere below `ADAPTER_ROOT`. Either copy the whole
folder to your Drive (default path below) or upload it as one ZIP when asked. Optional: the
supervisor's `utils.py` as `baselines/utils/utils.py` in the repository or next to the adapters;
otherwise an equivalent loader is used and this is printed.

**Run all is supported.** Part A takes a few minutes on any runtime. Set `RUN_SMOKE_TEST = False`
if you only want the geometry or have no A100.


**Cell 2**

Mount Drive, then clone or update the repository.


In [ ]:
# Cell 3
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

%cd /content
import os, shutil, sys
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis
assert os.path.isfile("scripts/assistant_adapter_geometry.py"), "Push scripts/assistant_adapter_geometry.py first."


**Cell 4**

Settings. `ADAPTER_ROOT` is searched recursively for the three expert folders.


In [ ]:
# Cell 5
from pathlib import Path
import json

PROJECT_ROOT = Path("/content/master-thesis").resolve()
RUN_TAG = "nb15_assistant_ppo_geometry"
GOALS = ["helpful", "harmless", "humor"]
EXPERT_SEED = 8888
ADAPTER_ROOT = Path("/content/drive/MyDrive/master-thesis-assistant")   # copy the supervisor's folder here
ADAPTER_ZIP_NAME = "assistant_ppo_adapters.zip"                         # alternative: upload one ZIP
EXTRACT_ROOT = Path("/content/assistant_adapters")
RESULTS_DIR = PROJECT_ROOT / "results" / RUN_TAG
DRIVE_OUT = Path("/content/drive/MyDrive/master-thesis-nb15") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

RUN_SMOKE_TEST = True
BASE_MODEL = "meta-llama/Llama-2-7b-hf"
MAX_NEW_TOKENS = 128
SMOKE_PROMPTS = [
    "How do I pick a lock?",
    "Tell me a joke about computers.",
    "What are some practical tips for staying healthy during winter?",
]
RESPONSE_SPLIT_DEFAULT = "\n\nAssistant:"
print(f"adapter root = {ADAPTER_ROOT}\nresults      = {RESULTS_DIR}")


**Cell 6**

Locate the three expert folders. If they are not below `ADAPTER_ROOT`, a ZIP of the supervisor's folder is requested and extracted safely.


In [ ]:
# Cell 7
import importlib.util, stat, zipfile

spec = importlib.util.spec_from_file_location("assistant_geometry", PROJECT_ROOT / "scripts/assistant_adapter_geometry.py")
assistant_geometry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(assistant_geometry)

def try_find(root):
    try:
        return assistant_geometry.find_adapter_dirs(root, GOALS, EXPERT_SEED)
    except FileNotFoundError as error:
        print(f"[not found] {error}")
        return None

def safe_extract(zip_path, target):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        total = sum(info.file_size for info in archive.infolist())
        assert total < 20e9, f"Unexpectedly large archive: {total / 1e9:.1f} GB"
        for info in archive.infolist():
            if (info.external_attr >> 16) & 0o170000 == stat.S_IFLNK:
                raise RuntimeError(f"Symlink in archive: {info.filename}")
            destination = (target / info.filename).resolve()
            if target != destination and target not in destination.parents:
                raise RuntimeError(f"Unsafe path in archive: {info.filename}")
        archive.extractall(target)

ADAPTER_DIRS = try_find(ADAPTER_ROOT) if ADAPTER_ROOT.exists() else None
if ADAPTER_DIRS is None:
    candidates = [p for p in (ADAPTER_ROOT / ADAPTER_ZIP_NAME, Path("/content") / ADAPTER_ZIP_NAME) if p.is_file()]
    if not candidates:
        from google.colab import files
        print("Please upload a ZIP of the supervisor's folder (structure intact).")
        uploaded = files.upload()
        candidates = [Path("/content") / name for name in uploaded if name.endswith(".zip")]
    assert candidates, "No adapter ZIP provided."
    safe_extract(candidates[0], EXTRACT_ROOT)
    ADAPTER_DIRS = try_find(EXTRACT_ROOT)
assert ADAPTER_DIRS is not None, "Expert folders <goal>_model or ppo_assistant_<goal>_s8888 were not found."
for goal, path in ADAPTER_DIRS.items():
    print(f"{goal:9s} -> {path}")


**Cell 8**

Part A: audit, geometry and floor certificate. The script stops with an explicit message if an adapter stores non-LoRA tensors, uses DoRA or `modules_to_save`, or if two goals point to identical files.


In [ ]:
# Cell 9
import numpy as np, pandas as pd
ADAPTER_PARENT = Path(os.path.commonpath([str(p) for p in ADAPTER_DIRS.values()]))
REPORT = assistant_geometry.run(ADAPTER_PARENT, RESULTS_DIR, goals=GOALS, seed=EXPERT_SEED)

print("Adapter configuration (helpful):")
print(json.dumps(REPORT["audits"]["helpful"]["config"], indent=2))
print(f"\nadapted modules: {REPORT['n_adapted_modules']} ({', '.join(REPORT['adapted_module_types'])}), "
      f"effective dimension d = {REPORT['effective_update_numel']:,}")
print("update norms:", {g: round(v, 3) for g, v in REPORT["update_norms"].items()})
for warning in REPORT["warnings"]:
    print("[warning]", warning)
for label in ("R_cos", "R_gram"):
    print(f"\n{label}")
    display(pd.read_csv(RESULTS_DIR / f"{label}_assistant.csv", index_col=0).round(4))


**Cell 10**

Reading the result. The primary matrix is `R_cos` (as in NB13/NB14). `R_gram` is shown because the supervisor's experts were not trained with equal sample sizes, so their norms may differ, and the floor of the Gram matrix can then differ from the cosine floor.


In [ ]:
# Cell 11
rows = []
for label in ("R_cos", "R_gram"):
    s = REPORT[label]
    rows.append({
        "matrix": label,
        **{f"cos/inner {k}": round(v, 4) for k, v in s["pairs"].items()},
        "negative off-diagonals": s["n_negative_offdiag"],
        "R^-1 1 (scaled to max |.| = 1)": np.round(np.asarray(s["inverse_R_ones"]) / np.max(np.abs(s["inverse_R_ones"])), 4).tolist(),
        "condition": round(s["condition_number"], 3),
        "near singular": s["near_singular"],
        "floor collapsed": f'{s["floor_lp"]["n_collapsed"]}/{s["floor_lp"]["n_preferences"]}',
        "redundant axis": s["redundant_axes_inverse_R_ones_nonpositive"] or "none",
    })
display(pd.DataFrame(rows).set_index("matrix").T)

s = REPORT["R_cos"]
if s["near_singular"]:
    verdict = ("R_cos is nearly singular (one expert is almost a combination of the others): "
               f"floor open for {len(s['floor_lp']['open_preferences'])} of {s['floor_lp']['n_preferences']} preferences, "
               "but interpret R^-1 1 with care.")
elif not s["inverse_R_ones_strictly_positive"]:
    verdict = (f"FLOOR OPENS on R_cos. Redundant axis: {s['redundant_axes_inverse_R_ones_nonpositive']}. "
               f"Open for {len(s['floor_lp']['open_preferences'])} of {s['floor_lp']['n_preferences']} preferences. "
               "This is the positive control: Cert can move away from p here.")
elif s["n_negative_offdiag"] > 0:
    verdict = ("Floor collapses on R_cos although some experts are negatively aligned: "
               "conflict alone does not open the floor, redundancy would be needed.")
else:
    verdict = "Floor collapses on R_cos with non-negative alignment: replication of the HelpSteer2 result."
print("VERDICT:", verdict)
if REPORT["R_gram"]["inverse_R_ones_strictly_positive"] != s["inverse_R_ones_strictly_positive"]:
    print("NOTE: R_gram and R_cos disagree on the floor. The norms differ enough to matter; report both.")
REPORT["verdict"] = verdict
(RESULTS_DIR / "assistant_geometry_report.json").write_text(json.dumps(REPORT, indent=2, default=str) + "\n")


**Cell 12**

Part B: pinned runtime for the smoke test (same stack as NB14). Skipped when `RUN_SMOKE_TEST = False`.


In [ ]:
# Cell 13
import importlib.metadata, subprocess
if RUN_SMOKE_TEST:
    os.environ["USE_TF"] = "0"
    os.environ["USE_TORCH"] = "1"
    packages = ["transformers==4.45.2", "tokenizers==0.20.3", "peft==0.13.2", "accelerate==1.1.1",
                "huggingface_hub==0.36.0", "protobuf==5.29.5", "sentencepiece", "safetensors"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "--only-binary=:all:", *packages], check=True)
    print({n: importlib.metadata.version(n) for n in ("transformers", "peft", "accelerate")})
else:
    print("[skip] smoke test disabled")


**Cell 14**

Part B: smoke test. Base model in bf16, embeddings resized to the checkpoint tokenizer before the adapter is attached, as in the supervisor's snippet. The three experts are attached as named adapters to one base, so the 7B model is loaded only once; `disable_adapter()` gives the base model's answer for comparison.


In [ ]:
# Cell 15
SMOKE = {}
if RUN_SMOKE_TEST:
    import torch
    from google.colab import userdata
    from huggingface_hub import login
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    assert torch.cuda.is_available(), "Smoke test needs a GPU (A100 recommended)."
    token = userdata.get("HF_TOKEN")
    assert token, "Add your Hugging Face token as Colab secret HF_TOKEN."
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)

    # The supervisor's utils.py, if available; otherwise an equivalent loader.
    utils_source = "fallback"
    for candidate in (PROJECT_ROOT, ADAPTER_PARENT, ADAPTER_ROOT):
        if (candidate / "baselines/utils/utils.py").is_file():
            sys.path.insert(0, str(candidate))
            try:
                from baselines.utils.utils import Instructions, load_main_tokenizer
                utils_source = str(candidate / "baselines/utils/utils.py")
            except Exception as error:
                print(f"[note] supervisor utils.py found but not importable ({error}); using fallback")
            break
    if utils_source == "fallback":
        # Shared Drive layout: utils.py lies flat next to the expert folders.
        flat = [c / "utils.py" for c in (ADAPTER_PARENT, ADAPTER_ROOT, EXTRACT_ROOT) if (c / "utils.py").is_file()]
        if flat:
            try:
                utils_spec = importlib.util.spec_from_file_location("supervisor_utils", flat[0])
                supervisor_utils = importlib.util.module_from_spec(utils_spec)
                utils_spec.loader.exec_module(supervisor_utils)
                Instructions = supervisor_utils.Instructions
                load_main_tokenizer = supervisor_utils.load_main_tokenizer
                utils_source = str(flat[0])
            except Exception as error:
                print(f"[note] {flat[0]} found but not importable ({error}); using fallback")
    if utils_source == "fallback":
        class Instructions:
            response_split = RESPONSE_SPLIT_DEFAULT
            @staticmethod
            def get_response(text):
                return text.split(Instructions.response_split)[-1].strip()
        def load_main_tokenizer(path):
            tok = AutoTokenizer.from_pretrained(path, use_fast=False)
            tok.pad_token = tok.eos_token
            return tok
    print(f"utils: {utils_source}; response split = {Instructions.response_split!r}")

    tokenizers = {goal: load_main_tokenizer(str(path)) for goal, path in ADAPTER_DIRS.items()}
    sizes = {goal: len(tok) for goal, tok in tokenizers.items()}
    assert len(set(sizes.values())) == 1, f"Tokenizer sizes differ between experts: {sizes}"
    tokenizer = tokenizers[GOALS[0]]
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map=0)
    base_vocab = model.get_input_embeddings().weight.shape[0]
    model.resize_token_embeddings(len(tokenizer))
    print(f"base vocab {base_vocab} -> tokenizer {len(tokenizer)} (resize is a no-op if equal)")
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIRS[GOALS[0]]), adapter_name=GOALS[0])
    for goal in GOALS[1:]:
        model.load_adapter(str(ADAPTER_DIRS[goal]), adapter_name=goal)
    model.eval()

    def generate(prompt):
        text = "\n\nHuman: " + prompt + " " + Instructions.response_split
        batch = tokenizer([text], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(batch["input_ids"], attention_mask=batch["attention_mask"],
                                 max_new_tokens=MAX_NEW_TOKENS, min_length=-1, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        return Instructions.get_response(tokenizer.batch_decode(out, skip_special_tokens=True)[0])

    with model.disable_adapter():
        SMOKE["base"] = {p: generate(p) for p in SMOKE_PROMPTS}
    for goal in GOALS:
        model.set_adapter(goal)
        SMOKE[goal] = {p: generate(p) for p in SMOKE_PROMPTS}

    for prompt in SMOKE_PROMPTS:
        print("\n" + "=" * 100 + f"\nPROMPT: {prompt}")
        for who in ["base", *GOALS]:
            print(f"\n--- {who} ---\n{SMOKE[who][prompt][:600]}")
    identical = {g: sum(SMOKE[g][p] == SMOKE["base"][p] for p in SMOKE_PROMPTS) for g in GOALS}
    print("\nanswers identical to the base model (of 3):", identical)
    (RESULTS_DIR / "smoke_test.json").write_text(json.dumps(
        {"utils_source": utils_source, "response_split": Instructions.response_split, "base_model": BASE_MODEL,
         "max_new_tokens": MAX_NEW_TOKENS, "tokenizer_size": len(tokenizer), "base_vocab": int(base_vocab),
         "answers": SMOKE, "identical_to_base": identical}, indent=2, ensure_ascii=False) + "\n")
    del model
    torch.cuda.empty_cache()


**Cell 16**

Export. The results folder is small and goes into the repository; a copy is kept on Drive.


In [ ]:
# Cell 17
from google.colab import files
zip_path = Path(f"/content/{RUN_TAG}_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_DIR.iterdir()):
        archive.write(path, f"results/{RUN_TAG}/{path.name}")
shutil.copy2(zip_path, DRIVE_OUT / zip_path.name)
print(f"{zip_path} ({zip_path.stat().st_size / 1e3:.0f} kB)\nVERDICT: {REPORT['verdict']}")
files.download(str(zip_path))


**Cell 18**

Repository status.


In [ ]:
# Cell 19
!git status --short
